In [2]:
# ─── CELL 1: IMPORTS ──────────────────────────────────────────────────────────

import csv
import os
import json
import time
import random
import pandas as pd
import sys
from datetime import datetime


print("✅ Imports done")

# ─── CELL 2: LOAD VAL_DF ──────────────────────────────────────────────────────

VAL_DF_PATH = "val_df.csv"  # ← update path
val_df = pd.read_csv(VAL_DF_PATH)
print(f"✅ val_df loaded: {len(val_df)} rows")
print(f"   Glucose range: {val_df['glucose'].min():.1f} – {val_df['glucose'].max():.1f} mg/dL")



✅ Imports done
✅ val_df loaded: 9067 rows
   Glucose range: 39.6 – 473.4 mg/dL


In [3]:
# ─── CELL 3: IMPORT AGENTS FROM YOUR MAIN NOTEBOOK ───────────────────────────
# Option A — if agents are defined in a .py file:
# Remove cached version if it exists
if 'capstone_agents_pipeline' in sys.modules:
    del sys.modules['capstone_agents_pipeline']
from capstone_agents_pipeline import run_main_with_safety, token_counter


/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.secretmanager_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.secretmanager_v1 past that date.
  warnings.warn(message, FutureWarning)


✅ Gemini API key setup complete from Secret Manager.
✅ Food API key setup complete from Secret Manager.
✅ ADK components imported successfully.
✅ Retry config for Agents created successfully.
✅ AlertAgent created.
✅ InsulinAgent created.
✅ search_food_by_carbs() created successfully
✅ MealAgent created.
✅ ExerciseAgent created.
✅ SafetyAgent created.
✅ Validation data loaded: 9047 rows, 53 features
✅ Prediction model loaded
✅ predict_glucose defined — row_number in, current + 4 future points out (mg/dL)
✅ Main_agent created.
✅ Formatter Agent created.


/opt/conda/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
# ─── CELL 4: ROW POOL SETUP ───────────────────────────────────────────────────

def find_rows(min_mgdl: float, max_mgdl: float, n: int = 20) -> list:
    """Both current glucose AND prediction must fall in [min_mgdl, max_mgdl]."""
    mask = (
        (val_df['glucose']    >= min_mgdl) & (val_df['glucose']    <= max_mgdl) #&
        #(val_df['prediction'] >= min_mgdl) & (val_df['prediction'] <= max_mgdl)
    )
    rows = val_df[mask].index.tolist()
    random.shuffle(rows)
    return rows[:n]

ROWS_NORMAL = find_rows(90,  150)
ROWS_HYPO   = find_rows(39,   69)
ROWS_HYPER  = find_rows(180, 400)

## Transition test cases

ROWS_TRANS_NH = [9047, 9048, 9049, 9050, 9051]
# glucose: 99, 105,  95, 110, 100  (Normal: 90–150)
# pred:    55,  62,  58,  65,  60  (Hypo: < 70)   → carb_rule = LOW
 
ROWS_TRANS_NR = [9052, 9053, 9054, 9055, 9056]
# glucose: 130, 140, 125, 135, 120  (Normal: 90–150)
# pred:    195, 205, 190, 200, 185  (Hyper: > 180) → carb_rule = HIGH
 
ROWS_TRANS_RN = [9057, 9058, 9059, 9060, 9061]
# glucose: 200, 185, 210, 240, 195  (Hyper: > 180)
# pred:    130, 125, 135, 140, 128  (Normal: 90–150) → carb_rule = NORMAL
 
ROWS_TRANS_HN = [9062, 9063, 9064, 9065, 9066]
# glucose:  62,  58,  65,  55,  60  (Hypo: < 70)
# pred:    100,  95, 105,  92,  98  (Normal: 90–150) → carb_rule = NORMAL

def safe_row(pool, idx, fallback_pool=None):
    if pool and idx < len(pool):
        return pool[idx]
    if fallback_pool:
        return fallback_pool[0]
    return ROWS_NORMAL[0]

print(f"   Normal  (90–150): {len(ROWS_NORMAL)} rows")
print(f"   Hypo    (39–69):  {len(ROWS_HYPO)} rows")
print(f"   Hyper   (180–400):{len(ROWS_HYPER)} rows")

# ── Day constants ─────────────────────────────────────────────────────────────
MON = "Monday"
TUE = "Tuesday"
WED = "Wednesday"
THU = "Thursday"
FRI = "Friday"
SAT = "Saturday"
SUN = "Sunday"

# ── Long-acting insulin templates ─────────────────────────────────────────────
LAI_NIGHTLY_9PM = "Yes, every night 9PM ET"
LAI_MORNING_7AM = "Yes, every morning 7AM ET"
LAI_NO          = "no"

# ── GLP-1 templates ───────────────────────────────────────────────────────────
GLP1_DAILY      = "daily"
GLP1_WEEKLY_SAT = "weekly on Saturdays"
GLP1_WEEKLY_MON = "weekly on Mondays"
GLP1_WEEKLY_FRI = "weekly on Fridays"
GLP1_PREMEAL    = "pre-meal"
GLP1_NO         = "no"

# ─── CELL 5: TEST CASE BUILDER ────────────────────────────────────────────────

def build_input(
    label,
    last_meal,
    current_time,           # format: "Saturday, 6:30 PM ET"
    row_number,
    diet                = "Non-Veg",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_NO,
    weight              = 75,
    height              = 1.65
):
    return {
        "label":      label,
        "row_number": row_number,
        "input": f"""
last_meal = {last_meal}
current_time = {current_time}
row_number = {row_number}
weight = {weight} kg
height = {height} m
diet = {diet}
usual_meal_times:
  breakfast = 7:00 AM ET
  lunch = 12:30 PM ET
  dinner = 7:00 PM ET
oral_medication = {oral_medication}
insulin = {insulin}
long_acting_insulin = {long_acting_insulin}
glp1 = {glp1}
"""
    }


   Normal  (90–150): 20 rows
   Hypo    (39–69):  20 rows
   Hyper   (180–400):20 rows


In [6]:
test_cases = []
 
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 1: NORMAL GLUCOSE — 10 cases
# Expected: carb_rule=NORMAL, balanced meal, no insulin correction
# ══════════════════════════════════════════════════════════════════════════════
 
test_cases.append(build_input(
    label               = "NORM-BRK-01 | Normal→Normal | Monday 30min BEFORE breakfast | oral+insulin+LAI",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{MON}, 6:30 AM ET",
    row_number          = safe_row(ROWS_NORMAL, 0),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "NORM-BRK-02 | Normal→Normal | Wednesday AT breakfast | no medication",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{WED}, 7:00 AM ET",
    row_number          = safe_row(ROWS_NORMAL, 1),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
test_cases.append(build_input(
    label               = "NORM-LCH-01 | Normal→Normal | Tuesday 30min BEFORE lunch | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 2),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "NORM-LCH-02 | Normal→Normal | Monday PAST lunch not taken | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 1:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 3),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "NORM-DIN-01 | Normal→Normal | Friday 30min BEFORE dinner | oral+insulin+LAI",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{FRI}, 6:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 4),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "NORM-DIN-02 | Normal→Normal | Sunday AT dinner | no medication",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SUN}, 7:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 5),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
test_cases.append(build_input(
    label               = "NORM-VEG-01 | Normal→Normal | Tuesday BEFORE lunch | Veg diet | oral+insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 6),
    diet                = "Veg",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "NORM-VGN-01 | Normal→Normal | Wednesday BEFORE lunch | Vegan diet | oral+insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{WED}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 7),
    diet                = "Vegan",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "NORM-INS-01 | Normal→Normal | Thursday BEFORE lunch | insulin ONLY no oral",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{THU}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 8),
    oral_medication     = "none",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "NORM-ORL-01 | Normal→Normal | Friday BEFORE dinner | oral ONLY no short-acting",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{FRI}, 6:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 9),
    oral_medication     = "pre-meal",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 2: HYPOGLYCEMIA — 12 cases
# Expected: carb_rule=LOW, fast-acting carbs, insulin=0, exercise=unsafe
# ══════════════════════════════════════════════════════════════════════════════
 
test_cases.append(build_input(
    label               = "HYPO-BRK-01 | Hypo→Hypo | Monday 30min BEFORE breakfast | oral+insulin+LAI",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{MON}, 6:30 AM ET",
    row_number          = safe_row(ROWS_HYPO, 0),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPO-LCH-01 | Hypo→Hypo | Friday 30min BEFORE lunch | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 1),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPO-LCH-02 | Hypo→Hypo | Thursday PAST lunch not taken | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{THU}, 1:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 2),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPO-DIN-01 | Hypo→Hypo | Thursday 30min BEFORE dinner | oral+insulin+LAI",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{THU}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 3),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPO-LAI-01 | Hypo→Hypo | Saturday 30min BEFORE 9PM LAI | critical — hypo + LAI due",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{SAT}, 8:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 4),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPO-LAI-02 | Hypo→Hypo | Monday 1hr AFTER 9PM LAI | hypo post-LAI high risk",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{MON}, 10:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 5),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPO-GATE-01 | Hypo→Hypo | Friday 3hrs BEFORE lunch | hypo overrides timing gate — meal required",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 9:30 AM ET",
    row_number          = safe_row(ROWS_HYPO, 6),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
test_cases.append(build_input(
    label               = "HYPO-GATE-02 | Hypo→Hypo | Thursday recently ate 30min ago | hypo overrides — still needs fast carbs",
    last_meal           = "Lunch at 12:00 PM ET",
    current_time        = f"{THU}, 12:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 7),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
test_cases.append(build_input(
    label               = "HYPO-VEG-01 | Hypo→Hypo | Monday BEFORE lunch | Veg diet | fast carbs must be veg-compliant",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 8),
    diet                = "Veg",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPO-VGN-01 | Hypo→Hypo | Tuesday BEFORE dinner | Vegan diet | fast carbs must be vegan-compliant",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{TUE}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 9),
    diet                = "Vegan",
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
test_cases.append(build_input(
    label               = "HYPO-GLP1-01 | Hypo→Hypo | Saturday BEFORE lunch | daily GLP-1 | hypo overrides insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{SAT}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 10),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_DAILY
))
 
test_cases.append(build_input(
    label               = "HYPO-GLP1-02 | Hypo→Hypo | Saturday BEFORE dinner | weekly GLP-1 DUE TODAY ✅ | hypo overrides insulin",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SAT}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 11),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))
 
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 3: HYPERGLYCEMIA — 13 cases
# Expected: carb_rule=HIGH, protein+veg only, insulin correction required
# ══════════════════════════════════════════════════════════════════════════════
 
test_cases.append(build_input(
    label               = "HYPR-BRK-01 | Hyper→Hyper | Wednesday 30min BEFORE breakfast | oral+insulin+LAI",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{WED}, 6:30 AM ET",
    row_number          = safe_row(ROWS_HYPER, 0),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-LCH-01 | Hyper→Hyper | Monday 30min BEFORE lunch | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 1),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-LCH-02 | Hyper→Hyper | Tuesday PAST lunch not taken | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 1:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 2),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-DIN-01 | Hyper→Hyper | Tuesday 30min BEFORE dinner | oral+insulin+LAI",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{TUE}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 3),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-WIN-01 | Hyper→Hyper | Friday 3hrs BEFORE lunch | outside window — no meal or insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 9:30 AM ET",
    row_number          = safe_row(ROWS_HYPER, 4),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-WIN-02 | Hyper→Hyper | Monday late night 11PM | outside all meal windows",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{MON}, 11:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 5),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-LAI-01 | Hyper→Hyper | Thursday 30min BEFORE 9PM LAI | hyper + LAI due",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{THU}, 8:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 6),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-VEG-01 | Hyper→Hyper | Monday BEFORE lunch | Veg diet | protein+veg only",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 7),
    diet                = "Veg",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-VGN-01 | Hyper→Hyper | Wednesday BEFORE dinner | Vegan diet | protein+veg only",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{WED}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 8),
    diet                = "Vegan",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-ORL-01 | Hyper→Hyper | Tuesday BEFORE lunch | oral ONLY no short-acting insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 9),
    oral_medication     = "pre-meal",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-INS-01 | Hyper→Hyper | Friday BEFORE dinner | insulin ONLY no oral",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{FRI}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 10),
    oral_medication     = "none",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "HYPR-GLP1-01 | Hyper→Hyper | Thursday BEFORE lunch | daily GLP-1 + oral + insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{THU}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 11),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_DAILY
))
 
test_cases.append(build_input(
    label               = "HYPR-GLP1-02 | Hyper→Hyper | Saturday BEFORE dinner | weekly GLP-1 DUE TODAY ✅ + oral + insulin",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SAT}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 12),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))
 
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 4: LONG-ACTING INSULIN — 4 cases
# ══════════════════════════════════════════════════════════════════════════════
 
test_cases.append(build_input(
    label               = "LAI-01 | Normal→Normal | Wednesday 30min BEFORE 9PM LAI | all medications",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{WED}, 8:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 10),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "LAI-02 | Normal→Normal | Tuesday AT 9PM LAI | long-acting ONLY no short-acting",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{TUE}, 9:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 11),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "LAI-03 | Normal→Normal | Friday 2hrs BEFORE 9PM LAI | outside window — no LAI alert",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{FRI}, 7:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 12),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "LAI-04 | Normal→Normal | Wednesday AT 7AM LAI | morning long-acting dose",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{WED}, 7:00 AM ET",
    row_number          = safe_row(ROWS_NORMAL, 13),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_MORNING_7AM
))
 
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 5: GLP-1 — 3 cases
# ══════════════════════════════════════════════════════════════════════════════
 
test_cases.append(build_input(
    label               = "GLP1-01 | Normal→Normal | Monday BEFORE lunch | daily GLP-1 + oral + insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 14),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_DAILY
))
 
test_cases.append(build_input(
    label               = "GLP1-02 | Normal→Normal | Saturday BEFORE lunch | weekly GLP-1 DUE TODAY ✅",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{SAT}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 15),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))
 
test_cases.append(build_input(
    label               = "GLP1-03 | Normal→Normal | Tuesday BEFORE lunch | weekly GLP-1 NOT TODAY ❌ (Sat scheduled)",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 16),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))
 
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 6: EDGE CASES — 8 cases
# ══════════════════════════════════════════════════════════════════════════════
 
test_cases.append(build_input(
    label               = "EDGE-01 | Normal→Normal | Tuesday recently ate 30min ago | no meal or insulin expected",
    last_meal           = "Lunch at 12:00 PM ET",
    current_time        = f"{TUE}, 12:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 17),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "EDGE-02 | Normal→Normal | Friday 3hrs BEFORE lunch | no meal or insulin expected",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 9:30 AM ET",
    row_number          = safe_row(ROWS_NORMAL, 18),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "EDGE-03 | Hypo→Hypo | Wednesday 3hrs BEFORE lunch | hypo overrides timing gate — meal required",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{WED}, 9:30 AM ET",
    row_number          = safe_row(ROWS_HYPO, 12),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
test_cases.append(build_input(
    label               = "EDGE-04 | Normal→Normal | Thursday 60min BEFORE 9PM LAI | exactly at window — LAI alert expected",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{THU}, 8:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 19),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "EDGE-05 | Normal→Normal | Friday 61min BEFORE 9PM LAI | just outside window — no LAI alert",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{FRI}, 7:59 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 0),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "EDGE-06 | Hypo→Hypo | Wednesday AT 9PM LAI | hypo + LAI due simultaneously",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{WED}, 9:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 13),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "EDGE-07 | Normal→Normal | Sunday BEFORE lunch | no medications at all — no alert expected",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{SUN}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 1),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO,
    glp1                = GLP1_NO
))
 
test_cases.append(build_input(
    label               = "EDGE-08 | Normal→Normal | Wednesday BEFORE dinner | weekly GLP-1 NOT TODAY ❌ — GLP-1 must be silent",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{WED}, 6:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 2),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))
 
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 7: TRANS — 20 cases (synthetic data, val_df indices 9047–9066)
#
# Expected agent behaviours:
#   TRANS-NH: alert fires | carb_rule=LOW | insulin=0 (must not worsen drop) | exercise=Avoid/Light
#   TRANS-NR: alert fires | carb_rule=HIGH | insulin 2–4 units (from predicted) | exercise=Moderate
#   TRANS-RN: alert fires | carb_rule=NORMAL (not HIGH) | insulin=0 | exercise=Moderate OK
#   TRANS-HN: immediate fast carbs for hypo NOW | carb_rule=NORMAL at meal | insulin=0 | exercise=Avoid
# ══════════════════════════════════════════════════════════════════════════════
 
# ── TRANS-NH: Normal→Hypo — 5 cases ──────────────────────────────────────────
 
test_cases.append(build_input(
    label               = "TRANS-NH-01 | Normal→Hypo | Tuesday 30min BEFORE breakfast | oral+insulin+LAI | glucose dropping fast — pred=55",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{TUE}, 6:30 AM ET",
    row_number          = safe_row(ROWS_TRANS_NH, 0),    # synth: glucose=99   model_pred≈55
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-NH-02 | Normal→Hypo | Wednesday 30min BEFORE lunch | oral+insulin+LAI | predicted dip below 70 — pred=62",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{WED}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_NH, 1),    # synth: glucose=105  model_pred≈62
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-NH-03 | Normal→Hypo | Friday 30min BEFORE lunch | oral only | borderline predicted hypo — pred=58",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_NH, 2),    # synth: glucose=95   model_pred≈58
    oral_medication     = "pre-meal",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-NH-04 | Normal→Hypo | Saturday 30min BEFORE dinner | no medication | glucose trending sharply down — pred=65",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SAT}, 6:30 PM ET",
    row_number          = safe_row(ROWS_TRANS_NH, 3),    # synth: glucose=110  model_pred≈65
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
test_cases.append(build_input(
    label               = "TRANS-NH-05 | Normal→Hypo | Sunday 30min BEFORE lunch | insulin+LAI | predicted to cross hypo threshold — pred=60",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{SUN}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_NH, 4),    # synth: glucose=100  model_pred≈60
    oral_medication     = "none",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
# ── TRANS-NR: Normal→Hyper — 5 cases ─────────────────────────────────────────
 
test_cases.append(build_input(
    label               = "TRANS-NR-01 | Normal→Hyper | Monday 30min BEFORE lunch | oral+insulin+LAI | post-carb spike predicted — pred=195",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_NR, 0),    # synth: glucose=130  model_pred≈195
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-NR-02 | Normal→Hyper | Wednesday 30min BEFORE dinner | oral+insulin+LAI | glucose trending sharply upward — pred=205",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{WED}, 6:30 PM ET",
    row_number          = safe_row(ROWS_TRANS_NR, 1),    # synth: glucose=140  model_pred≈205
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-NR-03 | Normal→Hyper | Thursday 30min BEFORE breakfast | insulin+LAI | rising trajectory requires correction — pred=190",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{THU}, 6:30 AM ET",
    row_number          = safe_row(ROWS_TRANS_NR, 2),    # synth: glucose=125  model_pred≈190
    oral_medication     = "none",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-NR-04 | Normal→Hyper | Friday 30min BEFORE lunch | oral only | predicted hyper — restrict carbs — pred=200",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_NR, 3),    # synth: glucose=135  model_pred≈200
    oral_medication     = "pre-meal",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-NR-05 | Normal→Hyper | Saturday 30min BEFORE dinner | no medication | steep predicted rise despite low-normal current — pred=185",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SAT}, 6:30 PM ET",
    row_number          = safe_row(ROWS_TRANS_NR, 4),    # synth: glucose=120  model_pred≈185
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
# ── TRANS-RN: Hyper→Normal — 5 cases ─────────────────────────────────────────
 
test_cases.append(build_input(
    label               = "TRANS-RN-01 | Hyper→Normal | Tuesday 30min BEFORE lunch | oral+insulin+LAI | glucose improving — balanced meal at meal time — pred=130",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_RN, 0),    # synth: glucose=200  model_pred≈130
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-RN-02 | Hyper→Normal | Thursday 30min BEFORE breakfast | oral+insulin+LAI | currently high but normalising — no correction — pred=125",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{THU}, 6:30 AM ET",
    row_number          = safe_row(ROWS_TRANS_RN, 1),    # synth: glucose=185  model_pred≈125
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-RN-03 | Hyper→Normal | Friday 30min BEFORE dinner | oral+insulin+LAI | rapid drop toward normal range — pred=135",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{FRI}, 6:30 PM ET",
    row_number          = safe_row(ROWS_TRANS_RN, 2),    # synth: glucose=210  model_pred≈135
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-RN-04 | Hyper→Normal | Monday 30min BEFORE lunch | insulin+LAI | high current normal predicted — carb rule must reflect prediction — pred=140",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_RN, 3),    # synth: glucose=240  model_pred≈140
    oral_medication     = "none",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-RN-05 | Hyper→Normal | Wednesday 30min BEFORE dinner | no medication | borderline hyper dropping to normal — pred=128",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{WED}, 6:30 PM ET",
    row_number          = safe_row(ROWS_TRANS_RN, 4),    # synth: glucose=195  model_pred≈128
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))
 
# ── TRANS-HN: Hypo→Normal — 5 cases ──────────────────────────────────────────
 
test_cases.append(build_input(
    label               = "TRANS-HN-01 | Hypo→Normal | Monday 30min BEFORE breakfast | oral+insulin+LAI | hypo recovering — fast carbs now, standard breakfast at meal — pred=100",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{MON}, 6:30 AM ET",
    row_number          = safe_row(ROWS_TRANS_HN, 0),    # synth: glucose=62   model_pred≈100
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-HN-02 | Hypo→Normal | Wednesday 30min BEFORE lunch | oral+insulin+LAI | currently hypo but glucose will normalise before lunch — pred=95",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{WED}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_HN, 1),    # synth: glucose=58   model_pred≈95
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-HN-03 | Hypo→Normal | Friday 30min BEFORE dinner | oral+insulin+LAI | hypoglycemia with improving trajectory — pred=105",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{FRI}, 6:30 PM ET",
    row_number          = safe_row(ROWS_TRANS_HN, 2),    # synth: glucose=65   model_pred≈105
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-HN-04 | Hypo→Normal | Saturday 30min BEFORE lunch | insulin+LAI | severe hypo recovering — standard meal at meal time — pred=92",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{SAT}, 12:00 PM ET",
    row_number          = safe_row(ROWS_TRANS_HN, 3),    # synth: glucose=55   model_pred≈92
    oral_medication     = "none",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
test_cases.append(build_input(
    label               = "TRANS-HN-05 | Hypo→Normal | Sunday 30min BEFORE dinner | oral+insulin+LAI | glucose rising from hypo to normal range — pred=98",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SUN}, 6:30 PM ET",
    row_number          = safe_row(ROWS_TRANS_HN, 4),    # synth: glucose=60   model_pred≈98
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))
 
# ─── SUMMARY ───────────────────────────────────────────────────────────────────
 
print(f"\n{'='*70}")
print(f"TOTAL TEST CASES: {len(test_cases)}")
print(f"{'='*70}")
 
groups = {
    "GROUP 1 — Normal glucose":      [tc for tc in test_cases if tc['label'].startswith('NORM')],
    "GROUP 2 — Hypoglycemia":        [tc for tc in test_cases if tc['label'].startswith('HYPO')],
    "GROUP 3 — Hyperglycemia":       [tc for tc in test_cases if tc['label'].startswith('HYPR')],
    "GROUP 4 — Long-acting insulin": [tc for tc in test_cases if tc['label'].startswith('LAI')],
    "GROUP 5 — GLP-1":               [tc for tc in test_cases if tc['label'].startswith('GLP1')],
    "GROUP 6 — Edge cases":          [tc for tc in test_cases if tc['label'].startswith('EDGE')],
    "GROUP 7 — Transitions":         [tc for tc in test_cases if tc['label'].startswith('TRANS')],
}
for name, cases in groups.items():
    print(f"  {name}: {len(cases)} cases")
 
print(f"\n── Row pool check ───────────────────────────────────────────────────────")
for pool_name, pool, needed in [
    ("ROWS_NORMAL",    ROWS_NORMAL,    20),
    ("ROWS_HYPO",      ROWS_HYPO,      14),
    ("ROWS_HYPER",     ROWS_HYPER,     13),
    ("ROWS_TRANS_NH",  ROWS_TRANS_NH,   5),
    ("ROWS_TRANS_NR",  ROWS_TRANS_NR,   5),
    ("ROWS_TRANS_RN",  ROWS_TRANS_RN,   5),
    ("ROWS_TRANS_HN",  ROWS_TRANS_HN,   5),
]:
    status = "✅" if len(pool) >= needed else "⚠️ "
    print(f"  {status} {pool_name}: {len(pool)} available, {needed} needed")
 


TOTAL TEST CASES: 70
  GROUP 1 — Normal glucose: 10 cases
  GROUP 2 — Hypoglycemia: 12 cases
  GROUP 3 — Hyperglycemia: 13 cases
  GROUP 4 — Long-acting insulin: 4 cases
  GROUP 5 — GLP-1: 3 cases
  GROUP 6 — Edge cases: 8 cases
  GROUP 7 — Transitions: 20 cases

── Row pool check ───────────────────────────────────────────────────────
  ✅ ROWS_NORMAL: 20 available, 20 needed
  ✅ ROWS_HYPO: 20 available, 14 needed
  ✅ ROWS_HYPER: 20 available, 13 needed
  ✅ ROWS_TRANS_NH: 5 available, 5 needed
  ✅ ROWS_TRANS_NR: 5 available, 5 needed
  ✅ ROWS_TRANS_RN: 5 available, 5 needed
  ✅ ROWS_TRANS_HN: 5 available, 5 needed


In [7]:
# ─── CELL 8: CSV HELPERS ──────────────────────────────────────────────────────

TEST_CSV_FILE = "test_results.csv"
TEST_CSV_HEADERS = [
    "run_timestamp",
    "test_id",
    "label",
    "row_number",
    "current_glucose",
    "glucose_at_meal_time",
    "glucose_category",
    "carb_rule",
    "meal_timing",
    "oral_medication",
    "insulin",
    "long_acting_insulin",
    "glp1",
    "diet",
    "elapsed_seconds",
    "input_tokens",          
    "output_tokens",         
    "is_safe",
    "attempts",
    "medication_alert",
    "insulin_units_recommended",
    "insulin_timing_recommended",
    "meal_recommended",
    "exercise_recommended",
    "violations",
    "status",
    "readable_output"
]


def reset_test_csv():
    if os.path.exists(TEST_CSV_FILE):
        os.remove(TEST_CSV_FILE)
        print(f"🗑️  Deleted old {TEST_CSV_FILE}")
    with open(TEST_CSV_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=TEST_CSV_HEADERS)
        writer.writeheader()
    print(f"✅ Created {TEST_CSV_FILE} with {len(TEST_CSV_HEADERS)} columns")

def get_glucose_category(glucose):
    try:
        g = float(glucose)
        if g < 70:   return "hypoglycemia"
        if g <= 150: return "normal"
        return "hyperglycemia"
    except Exception:
        return "unknown"

def extract_field(input_str, field_name):
    for line in input_str.splitlines():
        stripped = line.strip()
        if stripped.startswith(field_name):
            parts = stripped.split("=", 1)
            if len(parts) == 2:
                return parts[1].strip()
    return "unknown"

def extract_meal_timing(label):
    for keyword in ["BEFORE", "AFTER", "AT", "PAST"]:
        if keyword in label:
            for part in label.split("|"):
                if keyword in part:
                    return part.strip()
    return "unknown"

def parse_output_fields(result):
    current_glucose = glucose_at_meal_time = carb_rule = None
    insulin_units = insulin_timing = meal_recommended = None
    exercise_recommended = medication_alert = None
    is_safe = attempts = violations = None
    status = "unknown"

    if not isinstance(result, dict):
        return {k: None for k in [
            "current_glucose","glucose_at_meal_time","carb_rule",
            "is_safe","attempts","medication_alert",
            "insulin_units_recommended","insulin_timing_recommended",
            "meal_recommended","exercise_recommended",
            "violations","status"
        ]} | {"violations": "[]", "status": "unknown"}

    status   = result.get("status", "unknown")
    attempts = result.get("attempts", None)

    if status == "safe":
        is_safe  = True
        violations = "[]"
        output   = result.get("structured_output", {})
    else:
        is_safe  = False
        violations = json.dumps(result.get("violations", []))
        output   = result.get("last_output", {})

    if isinstance(output, dict):
        current_glucose      = output.get("current_glucose")
        glucose_at_meal_time = output.get("glucose_at_meal_time")
        carb_rule            = output.get("carb_rule")
        ins                  = output.get("insulin_recommendation", {}) or {}
        insulin_units        = ins.get("units")
        insulin_timing       = ins.get("timing")
        medication_alert     = str(output.get("medication_recommendation","") or "")
        meal_recommended     = str(output.get("meal_recommendation","") or "")
        ex = output.get("exercise_recommendation", {}) or {}
        if isinstance(ex, dict):
            exercise_recommended = (
                f"status={ex.get('status')} | "
                f"intensity={ex.get('max_intensity')} | "
                f"effect={ex.get('exercise_glucose_effect')} | "
                f"duration={ex.get('duration')}"
            )
        else:
            exercise_recommended = str(ex)

    return {
        "current_glucose":            current_glucose,
        "glucose_at_meal_time":       glucose_at_meal_time,
        "carb_rule":                  carb_rule,
        "is_safe":                    is_safe,
        "attempts":                   attempts,
        "medication_alert":           medication_alert,
        "insulin_units_recommended":  insulin_units,
        "insulin_timing_recommended": insulin_timing,
        "meal_recommended":           meal_recommended,
        "exercise_recommended":       exercise_recommended,
        "violations":                 violations,
        "status":                     status
    }

def append_test_result(test_id, tc, elapsed, result, token_counter=None):
    input_str = tc["input"]
    label     = tc["label"]
    parsed    = parse_output_fields(result)

    # ── Read token counts ──────────────────────────────────────────────────────
    input_tokens  = token_counter.input_tokens  if token_counter else None
    output_tokens = token_counter.output_tokens if token_counter else None

    # ── Formatter Agent output ────────────────────────────────────────────────
    # result["readable_output"] is the plain-text patient report
    # produced by FormatterAgent after a safe run.
    # For failed/errored runs it will be None.
    if isinstance(result, dict):
        readable_output = result.get("readable_output", None)
        if readable_output is None:
            # Run was not safe — store violations summary instead
            violations_list = result.get("violations", [])
            readable_output = (
                f"[NOT SAFE] {'; '.join(violations_list)}"
                if violations_list
                else f"[{result.get('status', 'unknown').upper()}]"
            )
    else:
        readable_output = str(result)

    current_glucose = parsed["current_glucose"]

    with open(TEST_CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=TEST_CSV_HEADERS)
        writer.writerow({
            "run_timestamp":              datetime.utcnow().isoformat(timespec="seconds") + "Z",
            "test_id":                    test_id,
            "label":                      label,
            "row_number":                 extract_field(input_str, "row_number"),
            "current_glucose":            current_glucose,
            "glucose_at_meal_time":       parsed["glucose_at_meal_time"],
            "glucose_category":           get_glucose_category(current_glucose),
            "carb_rule":                  parsed["carb_rule"],
            "meal_timing":                extract_meal_timing(label),
            "oral_medication":            extract_field(input_str, "oral_medication"),
            "insulin":                    extract_field(input_str, "insulin"),
            "long_acting_insulin":        extract_field(input_str, "long_acting_insulin"),
            "glp1":                       extract_field(input_str, "glp1"),
            "diet":                       extract_field(input_str, "diet"),
            "elapsed_seconds":            round(elapsed, 2),
            "input_tokens":               input_tokens,
            "output_tokens":              output_tokens,
            "is_safe":                    parsed["is_safe"],
            "attempts":                   parsed["attempts"],
            "medication_alert":           parsed["medication_alert"],
            "insulin_units_recommended":  parsed["insulin_units_recommended"],
            "insulin_timing_recommended": parsed["insulin_timing_recommended"],
            "meal_recommended":           parsed["meal_recommended"],
            "exercise_recommended":       parsed["exercise_recommended"],
            "violations":                 parsed["violations"],
            "status":                     parsed["status"],
            "readable_output":            readable_output,  
        })


In [8]:
# ─── CELL 9: TEST RUNNER ──────────────────────────────────────────────────────

async def run_all_tests(test_cases):
    reset_test_csv()
    summary = {"total": len(test_cases), "passed": 0, "failed": 0, "errored": 0}

    for i, tc in enumerate(test_cases):
        test_id = f"TC-{str(i+1).zfill(3)}"
        elapsed = 0
        print(f"\n{'='*60}")
        print(f"[{i+1}/{len(test_cases)}] {test_id}: {tc['label']}")
        print(f"{'='*60}")

        try:
            token_counter.reset()          # ← reset before each run

            start  = time.time()
            result = await run_main_with_safety(tc["input"])
            elapsed = time.time() - start

            print(f"   Tokens — input: {token_counter.input_tokens:,} | "
                  f"output: {token_counter.output_tokens:,}")

            if isinstance(result, dict) and result.get("status") == "failed":
                summary["failed"] += 1
                print(f"❌ Failed  | {elapsed:.2f}s | {result.get('violations')}")
            else:
                summary["passed"] += 1
                print(f"✅ Passed  | {elapsed:.2f}s")

            append_test_result(
                test_id,
                tc,
                elapsed,
                result,
                token_counter=token_counter    # ← pass counter
            )

        except Exception as e:
            elapsed = time.time() - start if elapsed == 0 else elapsed
            summary["errored"] += 1
            error_result = {"status": "error", "violations": [str(e)], "last_output": {}}
            append_test_result(
                test_id,
                tc,
                elapsed,
                error_result,
                token_counter=token_counter    # ← pass even on error
            )
            print(f"💥 Error   | {elapsed:.2f}s | {e}")

    print(f"\n{'='*60}")
    print(f"TEST RUN COMPLETE")
    print(f"  Total:      {summary['total']}")
    print(f"  ✅ Passed:  {summary['passed']}")
    print(f"  ❌ Failed:  {summary['failed']}")
    print(f"  💥 Errored: {summary['errored']}")
    print(f"  Results → {TEST_CSV_FILE}")
    print(f"{'='*60}")
    return summary


In [ ]:
# ─── CELL 10: RUN ─────────────────────────────────────────────────────────────

# Uncomment ONE of the options below:
MAX_RETRIES = 3
# Option A: run all test cases
summary = await run_all_tests(test_cases)

# Option B: run a single test case for quick debugging
# tc = test_cases[0]
# print(f"Running: {tc['label']}")
# result = await run_main_with_safety(tc["input"])
# print(result)

# Option C: run a specific group only
# group = [tc for tc in test_cases if tc['label'].startswith('HYPO')]
# summary = await run_all_tests(group)

🗑️  Deleted old test_results.csv
✅ Created test_results.csv with 27 columns

[1/70] TC-001: NORM-BRK-01 | Normal→Normal | Monday 30min BEFORE breakfast | oral+insulin+LAI

========== ATTEMPT 1 ==========

 ### Created new session: debug_session_id

User > {"user_input": "\nlast_meal = Dinner at 7:00 PM ET previous day\ncurrent_time = Monday, 6:30 AM ET\nrow_number = 4857\nweight = 75 kg\nheight = 1.65 m\ndiet = Non-Veg\nusual_meal_times:\n  breakfast = 7:00 AM ET\n  lunch = 12:30 PM ET\n  dinner = 7:00 PM ET\noral_medication = pre-meal\ninsulin = yes\nlong_acting_insulin = Yes, every night 9PM ET\nglp1 = no\n", "previous_output": null, "violations": []}
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-ebe2be8c-c557-4309-9400-9c2912e7de6e
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]    User ID: debug_user_id
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: Orchestrator_Agent
[logging_plugin]    User Content: t

/opt/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: AlertAgent | function_call: predict_glucose
[logging_plugin]    Token Usage - Input: 4478, Output: 182
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 4b6ab4f7-4af6-4416-ad97-71f024d7a710
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: AlertAgent | function_call: predict_glucose
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['AlertAgent', 'predict_glucose']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: AlertAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-0db4f214-407b-4afb-a728-7ef0d216546b
[logging_plugin]    Arguments: {'request': 'current_time: Monday, 6:30 AM ET. current_day: Monday. glp1_due_today: False. alert_scenario: upcoming_meal. usual_meal_times: breakfast=7:00 AM ET, lunch=12:30 PM ET, dinner=7:00 PM E

/opt/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: MealRecommenderAgent | function_call: ExerciseRecommenderAgent
[logging_plugin]    Token Usage - Input: 4602, Output: 201
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 5a709d1b-aefc-4d2b-8045-83736ef35508
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: MealRecommenderAgent | function_call: ExerciseRecommenderAgent
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['MealRecommenderAgent', 'ExerciseRecommenderAgent']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: MealRecommenderAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-bc33d634-586f-4c6f-9ef2-1b6ac07e04f1
[logging_plugin]    Arguments: {'request': 'glucose_at_meal_time: 102.6 mg/dL. predicted glucose trajectory: [103.4, 111.1, 117.6, 118.2] mg/dL. diet preference:

/opt/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: InsulinRecommenderAgent
[logging_plugin]    Token Usage - Input: 4838, Output: 22
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 975cc3c4-92b6-44ee-b109-47c85f405138
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: InsulinRecommenderAgent
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['InsulinRecommenderAgent']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: InsulinRecommenderAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-b65fc22d-ade0-468d-8521-630ae83e7d9b
[logging_plugin]    Arguments: {'request': '157.6'}
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-04887acf-2829-445e-8935-d62eca1c2b95
[logging_plugin]    Session ID: 938b5930-f26d-442c-ab97-f6c3775e7c7b
[logging_plugin]    User I

/opt/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: InsulinRecommenderAgent
[logging_plugin]    Token Usage - Input: 4857, Output: 33
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: c0428933-241e-44f9-b6d7-d69c21fa6ca1
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: InsulinRecommenderAgent
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['InsulinRecommenderAgent']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: InsulinRecommenderAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-ce41bbdc-b21e-40ce-bc6b-3468f4dcb811
[logging_plugin]    Arguments: {'request': 'glucose_at_meal_time: 91.8 mg/dL'}
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-17cbfd8e-cff8-41aa-aa65-be5a9802da17
[logging_plugin]    Session ID: 998ca0d3-4090-438d-a821-a6860230aee8